<a href="https://colab.research.google.com/github/chisamitsu/ita-dataanalytics/blob/main/s07-python-basico/s07-python-basico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# S07 – Python básico: funciones, estructuras de datos y bucles

Evidencias de ejecución – IT Academy, Data Analytics

| | |
|---|---|
| **Nombre** | Carlos Hisamitsu |
| **Fecha** | [DD/MM/AAAA] |
| **Repositorio GitHub** | https://github.com/chisamitsu/ita-dataanalytics/tree/main/s07-python-basico |

### Librerías

In [680]:
import re

import pandas as pd

# NIVEL 1

## Ejercicio 1 – Selección de personal

### Enunciado

> Tu treballes com a analista de dades en el departament de RRHH d'una empresa. S'ha obert una nova oferta laboral interna i l'empresa ha decidit que només les primeres 10 persones que s'inscriguin podran participar en el procés de selecció.

Per això, t'han encarregat desenvolupar una eina senzilla en Python que actuï com a formulari d'inscripció i sistema de classificació inicial. Aquest formulari ha de permetre introduir, per a cada candidat/a:

- Nom i Cognoms
- Telèfon de contacte
- Anys d'experiència dins de l'empresa
- Coneixement de Català (0–3 punts)
  - 0 = sense coneixement
  - 1 = fins a nivell B1
  - 2 = nivell B2
  - 3 = C1 o superior
- Domini d'Excel (0–3 punts)
  - 0 = nivell bàsic
  - 1 = ús de funcions
  - 2 = taules dinàmiques
  - 3 = Visual Basic i Macros
- Títol acadèmic (0–3 punts)
  - 0 = ESO
  - 1 = FP
  - 2 = Universitat
  - 3 = Màster o superior

Assegurant-se que tota la informació introduïda és correcta i entra dins els valors permesos. A partir d'aquestes dades, el programa haurà de calcular una puntuació total aplicant el pes de l'experiència (anys * 1.5) i sumant-hi els punts de català, excel i titulació, i a continuació classificar automàticament cada persona segons els llindars establerts per l'empresa:

>- Prioritat Alta → puntuació ≥ 12
>- Candidat adequat → 8 ≤ puntuació < 12
>- No prioritari → puntuació < 8

Cada vegada que algú complet a el formulari, s'ha de mostrar la seva puntuació individual. Quan ja s'hagin inscrit exactament deu persones, l'eina ha de deixar d'acceptar nous registres i generar un informe final en pantalla, ordenat primer segons la prioritat i, en cas d'empat, per la puntuació obtinguda. Aquest informe ha d'incloure el nom, el telèfon i la puntuació de cada candidat per tal que el departament de RRHH els pugui contactar ràpidament.

#### **Important**

Per resoldre aquest exercici, hauràs de:

- Desenvolupar un sistema modular amb les funcions següents:
  - obtenir_dades
  - validar_dades
  - calcular_puntuacio
  - classificar_candidat
  - mostrar_individual
  - mostrar_informe
  - main_seleccio # orquestra totes les funcions
- Escriure primer l'algoritme pas a pas, indicant clarament quines operacions farà cada funció.
- Escriure el codi de cada funció.

Per fer proves:

```
inputs_test = [
    "Laura Serra", "608112233", 6, 3, 3, 3,
    "Clara Gómez", "655221199", 3, 3, 3, 2,
    "Sergio López", "644112211", 3, 2, 2, 2,
    "Candidat Error", "611000000", 5, 7, 3, 2, # Català = 7 → NO VÀLID
    "Marta Ruiz", "699887766", 2, 3, 2, 1,
    "Marc Vidal", "612443322", 6, 3, 3, 3,
    "Júlia Romero", "677998877", 5, 3, 3, 2,
    "Pol Navarro", "699554433", 4, 3, 2, 2,
    "David Ferrer", "677665544", 2, 1, 1, 1,
    "Anna Paredes", "633221144", 1, 2, 1, 0,  
    "Anibal Paredes", "644272727", 1, 2, 1, 0,  
    "Núria Casals", "600112200", 8, 2, 3, 3 # 11è candidat     
]
```

### Algoritmo

#### Proceso (según el enunciado)
- Formulario:
  - obtener los datos
  - comprobar/validar los datos
- Calcular los puntos
- Definir los grupos de prioridad
- Mostrar los puntos calculados de cada candidato
- Máximo 10 inscripciones, después se cierra el formulario
- Mostrar el informe
  - ordenado por prioridad y después por puntos

#### Funciones

- **obtenir_dades** (`get_data`): recoge los datos de cada campo del formulario. Para asegurar que los datos son correctos, valida cada respuesta justo después de introducirla y vuelve a pedirla si no es válida. El usuario puede escribir "cancel" para descartar el candidato. Los datos del candidato se guardan en un diccionario.
- **validar_dades** (`validate_data`): función reutilizable que valida cada respuesta según los requisitos de su campo. Devuelve el valor (limpiado o convertido si hace falta, por ejemplo quitando espacios o convirtiendo números) o `None` si el dato es incorrecto, y también un mensaje de error, si lo hay.
- **calcular_puntuacio** (`candidate_score`): calcula los puntos con la fórmula definida.
- **classificar_candidat** (`candidate_tier`): asigna la categoría del candidato según su puntuación.
- **mostrar_individual** (`print_individual`): busca al candidato en la lista y muestra su puntuación y su categoría.
- **mostrar_informe** (`print_report`): muestra en pantalla el nombre, el teléfono y los puntos de los candidatos, ordenados por prioridad y puntos.
- **main_seleccio** (`run_registration`): orquesta las funciones para coordinar el proceso completo.

### Implementación

#### Preparación
Para facilitar el mantenimiento y poder definir las funciones de forma más dinámica, vamos a crear una lista con las preguntas, el tipo de respuesta esperada y las reglas de validación.

También definimos una (pseudo) constante con la cantidad máxima de candidatos y una lista vacía de candidatos como variable global (he considerado un diccionario de candidatos, pero podría haber candidatos con el mismo nombre, por lo que decidí usar una lista).

In [681]:
MAX_CANDIDATES = 10

PHONE_REGEX = r"^\+?\d{7,15}$"

# the list of candidates is a global variable
candidates = []

# define the list of questions, input type and data validation
form_questions = [
    {
        "id": "full_name",
        "input_label": "Nom i Cognoms",
        "type": "text",
        "len_min": 2,
        "len_max": 50,
        # at least one letter; letters, spaces, dots, apostrophes, hyphens and middle dot (l·l)
        "regex": r"^(?=.*[A-Za-zÀ-ÖØ-öø-ÿ])[A-Za-zÀ-ÖØ-öø-ÿ·.' -]{2,50}$",
        "regex_message": "El valor només pot contenir lletres, espais, punts, apòstrofs, guions i punt volat"
    },{
        "id": "phone",
        "input_label": "Telèfon de contacte",
        "type": "phone"
    },{
        "id": "seniority",
        "input_label": "Anys d'experiència dins de l'empresa",
        "type": "float",
        "range_min": 0,
        "range_max": 50
    },{
        "id": "level_catalan",
        "input_label": "Coneixement de Català",
        "type": "int",
        "range_min": 0,
        "range_max": 3,
        "tip": "0=Sense coneixement | 1=B1 | 2=B2 | 3=C1 o superior"
    },{
        "id": "level_excel",
        "input_label": "Domini d'Excel",
        "type": "int",
        "range_min": 0,
        "range_max": 3,
        "tip": "0=Bàsic | 1=Funcions | 2=Taules dinàmiques | 3=VBA i Macros"
    },{
        "id": "level_education",
        "input_label": "Títol acadèmic",
        "type": "int",
        "range_min": 0,
        "range_max": 3,
        "tip": "0=ESO | 1=FP | 2=Universitat | 3=Màster o superior"
    }
]

#### Funciones

##### **validate_data** (validar_dades)
Recibe como parámetro el diccionario "question" con los atributos esperados para el tipo de dato. Además, recibe el valor de respuesta del usuario para validar.

1. Uso el método ```question.get()``` para controlar posibles valores no existentes.
1. De acuerdo con el tipo de dato esperado (text, int, float, phone) la función aplica las validaciones definidas.
1. Si es posible, los datos se tratan para poder usarlos (elimina espacios en blanco al inicio o final del texto, convierte números, acepta la coma como separador decimal, etc.).
1. Si se encuentra algún error en la validación, se define un mensaje de error.
1. La función retorna el valor de respuesta formateado (si es posible) y el mensaje de error (si existe). Siempre habrá solamente un valor de retorno y el otro será ```None```.

In [682]:
# Validates a data input based on the type and requirements defined in the questions dictionary
def validate_data(question, answer):
  error_type = None
  error_message = None

  # get requirements, if they exist
  question_type = question.get("type")
  len_min = question.get("len_min", 1)   # text answers must not be empty by default
  len_max = question.get("len_max")
  range_min = question.get("range_min")
  range_max = question.get("range_max")
  regex = question.get("regex")
  regex_message = question.get("regex_message","El valor no és vàlid")

  # validate text type
  if question_type == "text":
    answer = answer.strip()
    # validate length, each bound is optional (len_min is 1 if not defined)
    if (len(answer) < len_min or (len_max is not None and len(answer) > len_max)):
      error_type = "length"
    # in case a regular expression was defined, then validate it
    elif (regex is not None and not re.match(regex, answer)):
      error_type = "regex"

  # validate integer type
  elif question_type == "int":
    try:
      answer = int(answer)
      # only validate range if both range_min and range_max are defined
      if (range_min is not None and range_max is not None and not range_min <= answer <= range_max):
        error_type = "range"
    except ValueError:
      error_type = "type int"

  # validate float type (a comma is accepted as decimal separator)
  elif question_type == "float":
    try:
      answer = float(answer.replace(",", "."))
      # only validate range if both range_min and range_max are defined
      if (range_min is not None and range_max is not None and not range_min <= answer <= range_max):
        error_type = "range"
    except ValueError:
      error_type = "type float"

  # validate phone type
  elif question_type == "phone":
    # clean up unnecessary characters that are common on phone info
    answer = answer.strip()
    answer = (answer.replace(" ", "").replace("-", ""))

    if not re.match(PHONE_REGEX, answer):
      error_type = "phone"

  # if any error was found, then clean the answer value as it is not valid
  if error_type is not None:
    answer = None

  # set the error message, if any error was detected
  if error_type == "length":
    if len_max is None:
      error_message = f"El valor ha de tenir com a mínim {len_min} caràcters"
    else:
      error_message = f"El valor ha de tenir entre {len_min} i {len_max} caràcters"
  elif error_type == "range":
    error_message = f"El valor ha de ser entre {range_min} i {range_max}"
  elif error_type == "type int":
    error_message = "El valor ha de ser un número enter"
  elif error_type == "type float":
    error_message = "El valor ha de ser un número (es poden usar decimals)"
  elif error_type == "phone":
    error_message = "El valor ha de ser un número de telèfon vàlid"
  elif error_type == "regex":
    error_message = regex_message

  return answer, error_message

##### **get_data** (obtenir_dades)
Para cada pregunta en la lista ```form_questions```:
1. Escribe la pregunta.
1. Si hay pista para la pregunta, la escribe debajo.
1. El usuario responde la pregunta.
1. Si la respuesta es "cancel", descarta el candidato y retorna ```None```.
1. Valida la respuesta de acuerdo con el tipo de dato (```validate_data()```).
1. Si hay algún error, muestra el mensaje y espera un nuevo valor. Repite hasta que el dato sea correcto.
1. Si no hay error, guarda el dato en el diccionario del candidato (```candidate```).

In [683]:
# Gets the candidate's data and returns it as a dictionary (None if the registration is cancelled)
def get_data():
  candidate = {}
  for question in form_questions:
    print(f"\n{question['input_label']}")
    tip = question.get("tip")
    if tip:
        print(f"{tip}")

    while True:
      answer = input("> ")

      if answer.strip().lower() == "cancel":
        return None

      answer, error_message = validate_data(question, answer)
      if error_message:
        print(error_message)
        print("Introdueix un valor vàlid o escriu 'cancel' per descartar el candidat")
      else:
        candidate[question['id']] = answer
        break

  return candidate

##### **candidate_score** (calcular_puntuacio)
Una función sencilla para calcular la puntuación del candidato.
1. Recibe como parámetro los datos del candidato.
1. Busca los valores necesarios para calcular.
1. Calcula con la fórmula ```(seniority * 1.5) + level_catalan + level_excel + level_education```.
1. Retorna el resultado.

In [684]:
def candidate_score(candidate):
  seniority = candidate.get("seniority",0)
  level_catalan = candidate.get("level_catalan",0)
  level_excel = candidate.get("level_excel",0)
  level_education = candidate.get("level_education",0)

  score = (seniority * 1.5) + level_catalan + level_excel + level_education
  return score

##### **candidate_tier** (classificar_candidat)
Una función sencilla que recibe como parámetro los puntos del candidato y retorna la clasificación.
1. Si puntos ≥ 12: Prioritat Alta
1. Si 8 ≤ puntos < 12: Candidat adequat
1. Si puntos < 8: No prioritari

In [685]:
def candidate_tier(score):
  tier = ""
  if score >= 12:
    tier = "Prioritat Alta"
  elif score >= 8:
    tier = "Candidat adequat"
  else:
    tier = "No prioritari"
  return tier

##### **print_individual** (mostrar_individual)
Recibe como parámetro el nombre del candidato y muestra en pantalla sus puntos, si lo encuentra.

La búsqueda se hace desde el final de la lista: así, si hay nombres repetidos, se muestra el candidato que se acaba de registrar.

In [686]:
def print_individual(candidate_name):
  # search from the end: the candidate just registered is the last one
  for candidate in reversed(candidates):
    if candidate["full_name"] == candidate_name:
      print(f"\nCandidat: {candidate_name}")
      print(f"Puntuació: {candidate['score']}")
      print(f"Prioritat: {candidate['tier']}")
      break
  return

##### **print_report** (mostrar_informe)
Muestra en pantalla los datos de los candidatos registrados, ordenados por prioridad y puntos. El ejercicio pide ordenar por prioridad y después por puntos; como la prioridad se calcula a partir de los puntos, ordenar por puntos es suficiente.

Muestra el nombre, el teléfono y los puntos. La columna numérica se alinea a la derecha.

In [687]:
def print_report():
  candidates.sort(key=lambda x: (x["score"]), reverse=True)

  # column widths: the minimum needed for the header and the longest value
  headers = ("Nom i Cognoms", "Telèfon", "Punts")
  alignments = ["<", "<", ">"]
  rows = [(c["full_name"], c["phone"], f"{c['score']:.1f}") for c in candidates]
  widths = [max(len(value) for value in column) for column in zip(headers, *rows)]
  line = "-" * (sum(widths) + 3 * (len(widths) - 1))

  print("\nInforme de candidats\n")
  print(line)
  print(" | ".join(f"{header:<{width}}" for header, width in zip(headers, widths)))
  print(line)

  for row in rows:
    print(" | ".join(f"{value:{align}{width}}" for value, width, align in zip(row, widths, alignments)))

  return

##### **run_registration** (main_seleccio)

Coordina la ejecución de la inscripción de candidatos.

- El número total de candidatos está definido en la variable global ```MAX_CANDIDATES```.
- Empieza mostrando el número del candidato que será registrado.
- Llama a la función para recopilar los datos del candidato ```get_data()```.
- Si los datos del candidato son correctos, calcula la puntuación y la prioridad, y guarda los datos en la lista de candidatos. Si el registro se cancela, el candidato se descarta y no cuenta para el máximo.
- Cuando llega al número máximo de candidatos, cierra las inscripciones y muestra el informe ordenado.

In [688]:
def run_registration():
  # create the candidates list
  global candidates
  candidates = []

  while len(candidates) < MAX_CANDIDATES:
    print("=" * 50)
    print(f"\nInscriu el candidat {len(candidates) + 1}")
    candidate = get_data()
    if candidate:
      candidate["score"] = candidate_score(candidate)
      candidate["tier"] = candidate_tier(candidate["score"])
      candidates.append(candidate)

      print_individual(candidate["full_name"])
      print()
    else:
      print("\nCandidat descartat\n")

  print("=" * 50)
  print("\nInscripcions tancades\n")
  print("=" * 50)

  print_report()
  print()
  print("=" * 50)
  return

### Ejecución

Para la prueba uso los datos de `inputs_test` a través de una función `input()` que va leyendo la lista, así se ve en pantalla lo que se "escribe" en cada pregunta. Cambié la fila de "Candidat Error": después del 7 (que da error de rango) añadí "cancel" para que se vea cómo se descarta el candidato y se pasa al siguiente. Con los datos originales, el formulario volvería a pedir el valor y se desordenaría el resto.

Se registran 10 candidatos (el descartado no cuenta), y "Núria Casals", que sería el 11º, ya no se pregunta porque las inscripciones se cierran. El informe sale ordenado por puntos.

In [689]:
inputs_test = [
    "Laura Serra", "608112233", 6, 3, 3, 3,
    "Clara Gómez", "655221199", 3, 3, 3, 2,
    "Sergio López", "644112211", 3, 2, 2, 2,
    "Candidat Error", "611000000", 5, 7, "cancel",   # 7 → error, then cancel
    "Marta Ruiz", "699887766", 2, 3, 2, 1,
    "Marc Vidal", "612443322", 6, 3, 3, 3,
    "Júlia Romero", "677998877", 5, 3, 3, 2,
    "Pol Navarro", "699554433", 4, 3, 2, 2,
    "David Ferrer", "677665544", 2, 1, 1, 1,
    "Anna Paredes", "633221144", 1, 2, 1, 0,
    "Anibal Paredes", "644272727", 1, 2, 1, 0,
    "Núria Casals", "600112200", 8, 2, 3, 3,         # 11th: never asked
]

test_values = iter(inputs_test)

# replaces the built-in input() only for this test run
def input(prompt=""):
    value = str(next(test_values))
    print(f"{prompt}{value}")
    return value

run_registration()

del input   # restores the built-in input()


Inscriu el candidat 1

Nom i Cognoms
> Laura Serra

Telèfon de contacte
> 608112233

Anys d'experiència dins de l'empresa
> 6

Coneixement de Català
0=Sense coneixement | 1=B1 | 2=B2 | 3=C1 o superior
> 3

Domini d'Excel
0=Bàsic | 1=Funcions | 2=Taules dinàmiques | 3=VBA i Macros
> 3

Títol acadèmic
0=ESO | 1=FP | 2=Universitat | 3=Màster o superior
> 3

Candidat: Laura Serra
Puntuació: 18.0
Prioritat: Prioritat Alta


Inscriu el candidat 2

Nom i Cognoms
> Clara Gómez

Telèfon de contacte
> 655221199

Anys d'experiència dins de l'empresa
> 3

Coneixement de Català
0=Sense coneixement | 1=B1 | 2=B2 | 3=C1 o superior
> 3

Domini d'Excel
0=Bàsic | 1=Funcions | 2=Taules dinàmiques | 3=VBA i Macros
> 3

Títol acadèmic
0=ESO | 1=FP | 2=Universitat | 3=Màster o superior
> 2

Candidat: Clara Gómez
Puntuació: 12.5
Prioritat: Prioritat Alta


Inscriu el candidat 3

Nom i Cognoms
> Sergio López

Telèfon de contacte
> 644112211

Anys d'experiència dins de l'empresa
> 3

Coneixement de Català
0=S

### Interpretación de los resultados

Se registraron los 10 candidatos. El de "Candidat Error" se descartó con "cancel" después del error con el valor "7", y "Núria Casals" no llegó a preguntarse porque ya estaban cerradas las inscripciones.

Cinco quedaron en Prioritat Alta (≥12), dos en Candidat adequat (≥8 y <12) y tres en No prioritari (<8). La experiencia es lo que más pesa en la puntuación, por eso Laura y Marc (6 años) están arriba con 18.0 (pero también tienen la puntuación máxima en catalán, Excel y título académico).

Hay dos empates (Laura y Marc, Anna y Anibal) y el informe los deja en el orden de registro.

## Ejercicio 2 – Segmentación por riesgo de carrito de compra

### Enunciado

> Treballes com a analista de dades al departament de Màrqueting Digital. L'empresa vol millorar l'efectivitat de les campanyes de recuperació de cistellas de compra abandonats i necessita identificar quins clients tenen més risc d'abandonament, quin segment de comportament presenten i quin missatge publicitari és el més adequat per a cadascun.

Disposes d'un conjunt de dades amb informació bàsica de 10 clients:

- id_client
- nom
- edat
- grup_edat
- codi_comportament

```
clients = [
    {"id_client": 1, "nom": "Laura Serra", "edat": 23, "grup_edat": "18-25", "codi_comportament": 4},
    {"id_client": 2, "nom": "Marc Vidal", "edat": 35, "grup_edat": "26-40", "codi_comportament": 3},
    {"id_client": 3, "nom": "Ana López", "edat": 29, "grup_edat": "26-40", "codi_comportament": 2},
    {"id_client": 4, "nom": "Joan Riera", "edat": 19, "grup_edat": "18-25", "codi_comportament": 2},
    {"id_client": 5, "nom": "Mónica Pérez", "edat": 47, "grup_edat": "41-60", "codi_comportament": 4},
    {"id_client": 6, "nom": "Luis García", "edat": 52, "grup_edat": "41-60", "codi_comportament": 1},
    {"id_client": 7, "nom": "Pilar Sánchez", "edat": 61, "grup_edat": "60+", "codi_comportament": 3},
    {"id_client": 8, "nom": "Eva Martín", "edat": 38, "grup_edat": "26-40", "codi_comportament": 0},
    {"id_client": 9, "nom": "Diego Romero", "edat": 24, "grup_edat": "18-25", "codi_comportament": 1},
    {"id_client": 10, "nom": "Núria Costa", "edat": 33, "grup_edat": "26-40", "codi_comportament": 4}
]
```

El departament de CRM ha definit manualment:

- Codis de comportament
  - 0 → Navega però no afegeix productes
  - 1 → Afegeix productes però no arriba al checkout
  - 2 → Arriba al checkout però no paga
  - 3 → Cistella de compra abandonat fa menys de 24 h
  - 4 → Cistella de compra abandonat fa més de 24 h
- Factors segons codi de comportament
  - Codi 0 → 0.10
  - Codi 1 → 0.25
  - Codi 2 → 0.50
  - Codi 3 → 0.75
  - Codi 4 → 0.90
- Factors segons grup d'edat
  - 18–25 → 1.20
  - 26–40 → 1.00
  - 41–60 → 0.85
  - 60+ → 0.70
- Fórmula del risc final
  - risc_final = factor codi * factor edat

Un cop processats els clients, has de ser capaç de respondre:

1. Quins clients s'han de contactar primer en una campanya de retargeting?
1. Quines diferències hi ha entre clients amb el mateix comportament però d'edats diferents? (Per exemple: un client jove que abandona la cistella de compra fa 24 h té el mateix risc que un de més edat?)
1. Quins clients presenten tan poc interès que no val la pena invertir-hi pressupost?
1. Quin client seria el més rendible de recuperar?

Interpretant el risc final i el segment.

#### Requisits tècnics del teu programa

Hauràs de construir un sistema modular utilitzant diverses funcions i, sobretot, sense utilitzar if/elif/else per fer la segmentació.

#### **Important**

Per resoldre aquest exercici, hauràs de:

- Desenvolupar un sistema modular amb les funcions següents:
  - S'ha d'implementar amb un diccionari de diccionaris, sense condicions.
  - processar_clients # s'ha d'alimentar de les funcions següents:
    - obtenir_segment_i_factor
    - obtenir_factor_edat
    - calcular_risc
  - mostrar_dataframe o mostrar_diccionari
  - mostrar_respostes
  - main_marketing # orquestra totes les funcions
- Escriure primer l'algoritme pas a pas, indicant clarament quines operacions farà cada funció.
- Escriure el codi de cada funció.

#### Resultat esperat

Un DataFrame o diccionari final amb:

- id_cliente
- nombre
- edad
- grupo_edad
- codigo_comportamiento
- segment
- missatge
- factor_codi
- factor_edat
- risc_final

I capacitat per respondre totes les preguntes de màrqueting demanades en una funció.

### Algoritmo

#### Proceso (según el enunciado)

A partir de un dataframe de clientes, ha de procesar los datos y calcular factores clave para definir la estrategia de marketing para distintos segmentos.

1. Convertir el código de comportamiento en factor correspondiente (sin if/elif/else)
1. Definir el siguiente factor por la edad (sin if/elif/else)
1. Calcular el riesgo
1. Guardar todo en un DataFrame
1. Mostrar el DataFrame con el resultado del cálculo de factor de riesgo.
1. Responder las 4 preguntas del departamento de marketing con información relevante

#### Funciones

- **obtenir_segment_i_factor** (`get_segment_and_factor`): busca el código en un diccionario y devuelve segmento, mensaje y factor.
- **obtenir_factor_edat** (`get_age_factor`): lo mismo pero con el grupo de edad.
- **calcular_risc** (`calculate_risk`): multiplica los dos factores.
- **processar_clients** (`process_clients`): recorre los clientes, llama a las tres funciones de arriba y devuelve una lista con una fila por cliente.
- **mostrar_dataframe** (`show_dataframe`): pasa la lista a DataFrame, la ordena por riesgo y la muestra.
- **mostrar_respostes** (`show_answers`): contesta las 4 preguntas.
- **main_marketing** (`run_segmentation`): llama a todo en orden.

### Implementación

#### Preparación

El segmento es la descripción que da el departamento de CRM para cada código, tal cual sale en el enunciado. El mensaje es el texto publicitario que se le enviaría al cliente. El enunciado no da ninguno, así que los propongo.

Los factores van en diccionarios de diccionarios para evitar los if.

Para las preguntas hay números que elijo yo: a partir de qué riesgo ya no vale la pena invertir (`MIN_RISK`), cuántos clientes contactar primero (`TOP_CLIENTS`) y qué código es el de los que llegaron al checkout (`CHECKOUT_CODE`).

Crear un diccionario para convertir el código de comportamiento en factor

| Code | Segment | Message | Factor |
|--|--|--|--|
|0|Navega però no afegeix productes|Descobreix els nostres productes més populars|0.10|
|1|Afegeix productes però no arriba al checkout|Has triat uns productes genials, et falta poc per tenir-los|0.25|
|2|Arriba al checkout però no paga|Estàs a punt de finalitzar la compra, t'enviem la comanda gratis|0.50|
|3|Cistella de compra abandonat fa menys de 24 h|Encara tens la cistella a punt, acaba la compra avui|0.75|
|4|Cistella de compra abandonat fa més de 24 h|Et trobem a faltar! Torna amb un 10% de descompte|0.90|

Crear un diccionario para convertir la edad en factor

|Age|Factor|
|--|--|
|18–25|1.20|
|26–40|1.00|
|41–60|0.85|
|60+|0.70|

In [690]:
TOP_CLIENTS = 3
MIN_RISK = 0.25
CHECKOUT_CODE = 2 # reached checkout but did not pay

clients = [
    {"id_client": 1, "nom": "Laura Serra", "edat": 23, "grup_edat": "18-25", "codi_comportament": 4},
    {"id_client": 2, "nom": "Marc Vidal", "edat": 35, "grup_edat": "26-40", "codi_comportament": 3},
    {"id_client": 3, "nom": "Ana López", "edat": 29, "grup_edat": "26-40", "codi_comportament": 2},
    {"id_client": 4, "nom": "Joan Riera", "edat": 19, "grup_edat": "18-25", "codi_comportament": 2},
    {"id_client": 5, "nom": "Mónica Pérez", "edat": 47, "grup_edat": "41-60", "codi_comportament": 4},
    {"id_client": 6, "nom": "Luis García", "edat": 52, "grup_edat": "41-60", "codi_comportament": 1},
    {"id_client": 7, "nom": "Pilar Sánchez", "edat": 61, "grup_edat": "60+", "codi_comportament": 3},
    {"id_client": 8, "nom": "Eva Martín", "edat": 38, "grup_edat": "26-40", "codi_comportament": 0},
    {"id_client": 9, "nom": "Diego Romero", "edat": 24, "grup_edat": "18-25", "codi_comportament": 1},
    {"id_client": 10, "nom": "Núria Costa", "edat": 33, "grup_edat": "26-40", "codi_comportament": 4}
]

# behavior code -> segment, message and factor (segment is the CRM description from the enunciado, message is my own proposal)
behavior_data = {
    0: {"segment": "Navega però no afegeix productes", "missatge": "Descobreix els nostres productes més populars", "factor_codi": 0.10},
    1: {"segment": "Afegeix productes però no arriba al checkout", "missatge": "Has triat uns productes genials, et falta poc per tenir-los", "factor_codi": 0.25},
    2: {"segment": "Arriba al checkout però no paga", "missatge": "Estàs a punt de finalitzar la compra, t'enviem la comanda gratis", "factor_codi": 0.50},
    3: {"segment": "Cistella de compra abandonat fa menys de 24 h", "missatge": "Encara tens la cistella a punt, acaba la compra avui", "factor_codi": 0.75},
    4: {"segment": "Cistella de compra abandonat fa més de 24 h", "missatge": "Et trobem a faltar! Torna amb un 10% de descompte", "factor_codi": 0.90}
}

age_factors = {
    "18-25": 1.20,
    "26-40": 1.00,
    "41-60": 0.85,
    "60+": 0.70
}

#### Funciones

##### **get_segment_and_factor** (obtenir_segment_i_factor)

Busca el código en `behavior_data` y devuelve su diccionario (segmento, mensaje y factor).

In [691]:
def get_segment_and_factor(code):
  return behavior_data[code]

##### **get_age_factor** (obtenir_factor_edat)

Busca el grupo de edad en `age_factors` y devuelve su factor.

In [692]:
def get_age_factor(age_group):
  return age_factors[age_group]

##### **calculate_risk** (calcular_risc)

Multiplica los dos factores. Redondeo a 4 decimales.

In [693]:
def calculate_risk(factor_code, factor_age):
  return round(factor_code * factor_age, 4)

##### **process_clients** (processar_clients)

Para cada cliente:
1. Busca el segmento con su código de comportamiento.
1. Busca el factor de su grupo de edad.
1. Calcula el riesgo final.
1. Guarda una fila (diccionario) con las columnas que pide el enunciado.

|id_client|nom|edat|grup_edat|codi_comportament|segment|missatge|factor_codi|factor_edat|risc_final|
|--|--|--|--|--|--|--|--|--|--|

In [694]:
def process_clients(clients):
  results = []
  for client in clients:
    behavior = get_segment_and_factor(client["codi_comportament"])
    factor_age = get_age_factor(client["grup_edat"])

    results.append({
      "id_client": client["id_client"],
      "nom": client["nom"],
      "edat": client["edat"],
      "grup_edat": client["grup_edat"],
      "codi_comportament": client["codi_comportament"],
      "segment": behavior["segment"],
      "missatge": behavior["missatge"],
      "factor_codi": behavior["factor_codi"],
      "factor_edat": factor_age,
      "risc_final": calculate_risk(behavior["factor_codi"], factor_age)
    })
  return results

##### **show_dataframe** (mostrar_dataframe)

Pasa la lista a DataFrame, la ordena de más a menos riesgo, la muestra y la devuelve para usarla en las respuestas.

In [695]:
def show_dataframe(results):
  df = pd.DataFrame(results).sort_values("risc_final", ascending=False).reset_index(drop=True)
  display(df)
  return df

##### **show_answers** (mostrar_respostes)

Responde las 4 preguntas con la tabla ya ordenada. Para recorrer las filas con un `for` las paso a lista de diccionarios con `to_dict("records")`.

1. Los clientes con más riesgo, con `nlargest`.
2. Agrupo los clientes por código de comportamiento (`groupby`), y en cada grupo comparo el más joven con el mayor. Si solo hay un cliente en el grupo, lo salto.
3. Filtro la tabla con `df["risc_final"] < MIN_RISK`, que se queda solo con las filas por debajo del mínimo.
4. Hago un filtro parecido para quedarme con los que llegaron al checkout (código 2), y de esos cojo el de más riesgo (`idxmax`). Los códigos 3 y 4 (carrito abandonado) no los incluyo porque no se sabe si llegaron al checkout.

In [696]:
def show_answers(df):
  print("1. Clients a contactar primer (retargeting):")
  top_clients = df.nlargest(TOP_CLIENTS, "risc_final").to_dict("records")
  for client in top_clients:
    print(f"   {client['nom']} (risc {client['risc_final']})")

  print("\n2. Mateix comportament i edats diferents:")
  rows = []
  for code, group in df.groupby("codi_comportament"):
    if len(group) < 2:
      continue
    group = group.sort_values("edat")
    young, old = group.iloc[0], group.iloc[-1]
    rows.append({
      "Codi": code,
      "Més jove": young["nom"], "Edat jove": young["edat"], "Risc jove": young["risc_final"],
      "Més gran": old["nom"], "Edat gran": old["edat"], "Risc gran": old["risc_final"],
      "Diferència": round(young["risc_final"] - old["risc_final"], 4),
      "% més risc": round((young["risc_final"] / old["risc_final"] - 1) * 100)
    })
  display(pd.DataFrame(rows).set_index("Codi"))

  print(f"\n3. Clients amb risc inferior a {MIN_RISK} (no val la pena invertir-hi):")
  low_risk = df[df["risc_final"] < MIN_RISK].to_dict("records")
  for client in low_risk:
    print(f"   {client['nom']} (risc {client['risc_final']})")

  checkout = df[df["codi_comportament"] == CHECKOUT_CODE]
  best = checkout.loc[checkout["risc_final"].idxmax()]
  print(f"\n4. Client més rendible de recuperar: {best['nom']} ({best['edat']} anys)")
  print(f"   Segment: {best['segment']}, risc {best['risc_final']}")
  print("   Motiu: ha arribat al checkout, així que estava a prop de comprar")
  print(f"   Missatge: {best['missatge']}")


##### **run_segmentation** (main_marketing)

Procesa los clientes, muestra la tabla y responde las preguntas.

In [697]:
def run_segmentation(clients):
  results = process_clients(clients)
  print("Taula de clients (ordenada per risc):")
  df = show_dataframe(results)
  print("\n" + "-" * 60)
  print("Respostes de màrqueting:\n")
  show_answers(df)

### Ejecución

In [698]:
run_segmentation(clients)

Taula de clients (ordenada per risc):


,id_client,nom,edat,grup_edat,codi_comportament,segment,missatge,factor_codi,factor_edat,risc_final
0,1,Laura Serra,23,18-25,4,Cistella de compra abandonat fa més de 24 h,Et trobem a faltar! Torna amb un 10% de descompte,0.90,1.20,1.0800
1,10,Núria Costa,33,26-40,4,Cistella de compra abandonat fa més de 24 h,Et trobem a faltar! Torna amb un 10% de descompte,0.90,1.00,0.9000
2,5,Mónica Pérez,47,41-60,4,Cistella de compra abandonat fa més de 24 h,Et trobem a faltar! Torna amb un 10% de descompte,0.90,0.85,0.7650
3,2,Marc Vidal,35,26-40,3,Cistella de compra abandonat fa menys de 24 h,"Encara tens la cistella a punt, acaba la compr...",0.75,1.00,0.7500
4,4,Joan Riera,19,18-25,2,Arriba al checkout però no paga,"Estàs a punt de finalitzar la compra, t'enviem...",0.50,1.20,0.6000
5,7,Pilar Sánchez,61,60+,3,Cistella de compra abandonat fa menys de 24 h,"Encara tens la cistella a punt, acaba la compr...",0.75,0.70,0.5250
6,3,Ana López,29,26-40,2,Arriba al checkout però no paga,"Estàs a punt de finalitzar la compra, t'enviem...",0.50,1.00,0.5000
7,9,Diego Romero,24,18-25,1,Afegeix productes però no arriba al checkout,"Has triat uns productes genials, et falta poc ...",0.25,1.20,0.3000
8,6,Luis García,52,41-60,1,Afegeix productes però no arriba al checkout,"Has triat uns productes genials, et falta poc ...",0.25,0.85,0.2125
9,8,Eva Martín,38,26-40,0,Navega però no afegeix productes,Descobreix els nostres productes més populars,0.10,1.00,0.1000



------------------------------------------------------------
Respostes de màrqueting:

1. Clients a contactar primer (retargeting):
   Laura Serra (risc 1.08)
   Núria Costa (risc 0.9)
   Mónica Pérez (risc 0.765)

2. Mateix comportament i edats diferents:


,Més jove,Edat jove,Risc jove,Més gran,Edat gran,Risc gran,Diferència,% més risc
Codi,,,,,,,,
1,Diego Romero,24,0.30,Luis García,52,0.2125,0.0875,41
2,Joan Riera,19,0.60,Ana López,29,0.5000,0.1000,20
3,Marc Vidal,35,0.75,Pilar Sánchez,61,0.5250,0.2250,43
4,Laura Serra,23,1.08,Mónica Pérez,47,0.7650,0.3150,41



3. Clients amb risc inferior a 0.25 (no val la pena invertir-hi):
   Luis García (risc 0.2125)
   Eva Martín (risc 0.1)

4. Client més rendible de recuperar: Joan Riera (19 anys)
   Segment: Arriba al checkout però no paga, risc 0.6
   Motiu: ha arribat al checkout, així que estava a prop de comprar
   Missatge: Estàs a punt de finalitzar la compra, t'enviem la comanda gratis


### Interpretación de los resultados

1. Los que más riesgo tienen son los que abandonaron el carrito hace más de un día (Laura, Núria y Mónica), así que esos tres clientes son los primeros a contactar.

2. La edad también influye: con el mismo comportamiento el más joven siempre tiene más riesgo, y la diferencia es mayor cuanto más alto es el código de comportamiento. Un joven que abandona hace más de 24 h no tiene el mismo riesgo que uno mayor.

3. Luis y Eva quedan por debajo de 0.25 y no creo que compense invertir en ellos (el 0.25 lo elegí yo, el enunciado no da ningún límite).

4. Como más rentable me quedo con Joan Riera. Laura tiene más riesgo, pero Joan llegó al checkout y estaba más cerca de comprar. Los de los códigos 3 y 4 (carrito abandonado) quizá también llegaron al checkout, pero el enunciado no lo dice, así que solo cuento el código 2. Tampoco sé cuánto vale cada carrito, así que es una suposición.

# NIVEL 2

## Ejercicio 1 – Gestión de riesgo y priorización de pacientes

### Enunciado

> Treballes com a analista de dades en un centre mèdic especialitzat. Diversos pacients han enviat els seus símptomes a través d'un formulari online, però les dades arriben en un format desordenat i poc estructurat, i l'equip mèdic necessita que els organitzis per poder prioritzar l'ordre d'atenció segons la gravetat.

A continuació tens una llista amb informació real procedent del sistema:

```
pacients = [
    "Maria|45|tos, Fiebre Alta, dolor_pecho",
    "Luis|33|dolor_cabeza,faTiga",
    "Sara|67|fiebre alta, dificultad RESPIRAR, tos",
    "Jordi|52|fatiga , tos",
    "Anna|29|fiebre alta,dolor_cabeza"
]
```

El departament mèdic també t'ha proporcionat el següent diccionari de nivells de risc per símptoma:

```
nivells_risc = {
    "fiebre_alta": 3,
    "dificultad_respirar": 5,
    "dolor_pecho": 4,
    "tos": 1,
    "fatiga": 1,
    "dolor_cabeza": 1
}
```

#### Objectius

L'objectiu és transformar aquestes dades en informació útil per als metges. A partir dels registres en brut, has de:

- Convertir cada string en una estructura neta (diccionari o similar).
- Separar i normalitzar la llista de símptomes:
  - minúscules
  - eliminar espais
  - dividir correctament per comes
- Assignar un risc a cada símptoma utilitzant el diccionari nivells_risc.
- Calcular el risc total de cada pacient sumant els valors.
- Ordenar els pacients de major a menor risc per determinar la prioritat d'atenció.

#### **Important**

En aquest Nivell 2 ja no et proporcionem la llista de funcions que has de crear. Ets tu qui ha de plantejar quines funcions necessites i com modularitzar el procés.

Tot i així, es recomana que:

- La funció principal orquestri tots els passos.
- Cada subfunció faci una sola tasca concreta (neteja, conversió, càlcul, ordenació...).
- La lògica estigui separada de manera neta i coherent.

#### Resultat esperat

Un DataFrame o un diccionari amb:

- nom
- edat
- símptomes normalitzats
- risc_total

I un apartat final amb els pacients ordenats per prioritat (més risc → primer).

### Algoritmo

#### Proceso
- Por cada registro:
  - separar nombre, edad y síntomas (van separados por `|`)
  - limpiar los síntomas: minúsculas, sin espacios de más y separados por comas
- Sumar el riesgo de cada síntoma con `nivells_risc`
- Mostrar la tabla con nombre, edad, síntomas y riesgo total
  |nom|edat|simptomes normalitzats|risc_total|
  |--|--|--|--|

- Ordenar de más a menos riesgo para saber a quién atender primero

#### Funciones

- **normalize_symptoms**: limpia el texto de síntomas y devuelve una lista.
- **parse_patient**: pasa un registro a diccionario (nombre, edad, síntomas).
- **total_risk**: suma el riesgo de los síntomas.
- **sort_by_priority**: ordena los pacientes por riesgo.
- **show_priority**: muestra el orden de atención.
- **run_triage**: lo junta todo.

### Implementación

#### Preparación
Los datos son los del enunciado.

En el diccionario los síntomas llevan guion bajo (`fiebre_alta`), pero en los registros pueden venir con espacio (`fiebre alta`). Por eso al normalizar cambio los espacios por `_`. Los síntomas también deben estar en minúsculas.

In [699]:
pacients = [
    "Maria|45|tos, Fiebre Alta, dolor_pecho",
    "Luis|33|dolor_cabeza,faTiga",
    "Sara|67|fiebre alta, dificultad RESPIRAR, tos",
    "Jordi|52|fatiga , tos",
    "Anna|29|fiebre alta,dolor_cabeza"
]

nivells_risc = {
    "fiebre_alta": 3,
    "dificultad_respirar": 5,
    "dolor_pecho": 4,
    "tos": 1,
    "fatiga": 1,
    "dolor_cabeza": 1
}

#### Funciones

##### **normalize_symptoms**

1. Separa el texto por comas.
1. A cada síntoma le quita los espacios de los lados, lo pasa a minúsculas y cambia los espacios de dentro por `_`.

In [700]:
def normalize_symptoms(symptoms):
  return [symptom.strip().lower().replace(" ", "_") for symptom in symptoms.split(",")]

##### **parse_patient**

Separa el registro por `|` y devuelve un diccionario con el nombre, la edad (como número) y los síntomas ya normalizados.

In [701]:
def parse_patient(patient):
  name, age, symptoms = patient.split("|")
  return {
    "nom": name,
    "edat": int(age),
    "simptomes_normalitzats": normalize_symptoms(symptoms)
  }

##### **total_risk**

Suma el riesgo de cada síntoma. Si alguno no está en el diccionario cuenta 0, para que no falle.

In [702]:
def total_risk(symptoms,levels):
  return sum(levels.get(symptom,0) for symptom in symptoms)

##### **sort_by_priority**

Ordena de más a menos riesgo. Si dos pacientes empatan se quedan como estaban.

In [703]:
def sort_by_priority(patients):
  return sorted(patients, key=lambda patient: patient["risc_total"], reverse=True)

##### **show_priority**

Imprime la lista numerada con el orden de atención.

In [704]:
def show_priority(patients):
  print("Prioritat d'atenció:")
  for i, patient in enumerate(patients, 1):
    print(f"{i}. {patient['nom']} (risc: {patient['risc_total']})")

##### **run_triage**
1. Convierte cada registro en diccionario y le añade el riesgo total.
1. Muestra la tabla.
1. Ordena y muestra la prioridad.

In [705]:
def run_triage(pacients, nivells_risc):
  patients = [parse_patient(patient) for patient in pacients]
  for patient in patients:
    patient["risc_total"] = total_risk(patient["simptomes_normalitzats"], nivells_risc)

  df = pd.DataFrame(patients)
  display(df)

  print("\n" + "-" * 60)
  show_priority(sort_by_priority(patients))

### Ejecución

In [706]:
run_triage(pacients, nivells_risc)

,nom,edat,simptomes_normalitzats,risc_total
0,Maria,45,"[tos, fiebre_alta, dolor_pecho]",8
1,Luis,33,"[dolor_cabeza, fatiga]",2
2,Sara,67,"[fiebre_alta, dificultad_respirar, tos]",9
3,Jordi,52,"[fatiga, tos]",2
4,Anna,29,"[fiebre_alta, dolor_cabeza]",4



------------------------------------------------------------
Prioritat d'atenció:
1. Sara (risc: 9)
2. Maria (risc: 8)
3. Anna (risc: 4)
4. Luis (risc: 2)
5. Jordi (risc: 2)


### Interpretación de los resultados

Sara es la prioritaria con riesgo 9 (tiene dificultad para respirar, que pesa 5). Después va Maria con 8 y Anna con 4. Luis y Jordi empatan a 2, así que salen en el orden en que venían.

Sin normalizar, cosas como "Fiebre Alta" o "dificultad RESPIRAR" no coincidirían con las claves del diccionario.